# Sentiment Analysis using RNN and LSTM (SST-2 Dataset)
Date: 1/04/2026

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_df = pd.read_parquet('/content/drive/MyDrive/sst2_train.parquet')
val_df = pd.read_parquet('/content/drive/MyDrive/sst2_valid.parquet')

In [ ]:
#Dividing Test and Training Sets
test_df = train_df.sample(n=5000, random_state=42)
train_df = train_df.drop(test_df.index)

# **Pre-Processing**

In [ ]:
#Cleaning Texts
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

for df in [train_df, val_df, test_df]:
    df['sentence'] = df['sentence'].apply(clean_text)

#Converts words to numbers
def build_vocab(sentences, max_vocab=10000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.split())

    vocab = {"<PAD>":0, "<UNK>":1}
    for word, _ in counter.most_common(max_vocab-2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df['sentence'])

#Encoding
def encode(sentence, vocab, max_len=50):
    tokens = sentence.split()
    seq = [vocab.get(word, vocab["<UNK>"]) for word in tokens]
    seq = seq[:max_len]
    seq += [0] * (max_len - len(seq))
    return seq

max_len = 50

X_train = [encode(s, vocab, max_len) for s in train_df['sentence']]
X_val = [encode(s, vocab, max_len) for s in val_df['sentence']]
X_test = [encode(s, vocab, max_len) for s in test_df['sentence']]

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = torch.tensor(X_train).to(device)
y_train = torch.tensor(y_train).to(device)

X_val = torch.tensor(X_val).to(device)
y_val = torch.tensor(y_val).to(device)

X_test = torch.tensor(X_test).to(device)
y_test = torch.tensor(y_test).to(device)

train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# **Model Training**

In [ ]:
#RNN Model
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.dropout(out)
        return torch.sigmoid(self.fc(out))

In [ ]:
#LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return torch.sigmoid(self.fc(out))

In [ ]:
#Training Models
def train_model(model, loader, epochs=20):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        total_loss = 0

        for X_batch, y_batch in loader:   # 👈 THIS is the key change
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch.float())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

#Evaluating Models using accuracy, precision, recall and f1-score
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X).squeeze()
        preds = (preds > 0.5).int()

    acc = accuracy_score(y.cpu(), preds.cpu())
    prec = precision_score(y.cpu(), preds.cpu())
    rec = recall_score(y.cpu(), preds.cpu())
    f1 = f1_score(y.cpu(), preds.cpu())

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")

In [ ]:
vocab_size = len(vocab)

# RNN
rnn_model = RNNModel(vocab_size)
train_model(rnn_model, train_loader)
print("RNN Results:")
evaluate(rnn_model, X_test, y_test)

Epoch 1, Loss: 0.6916
Epoch 2, Loss: 0.6878
Epoch 3, Loss: 0.6873
Epoch 4, Loss: 0.6877
Epoch 5, Loss: 0.6898
Epoch 6, Loss: 0.6886
Epoch 7, Loss: 0.6891
Epoch 8, Loss: 0.6891
Epoch 9, Loss: 0.6889
Epoch 10, Loss: 0.6886
Epoch 11, Loss: 0.6885
Epoch 12, Loss: 0.6887
Epoch 13, Loss: 0.6885
Epoch 14, Loss: 0.6887
Epoch 15, Loss: 0.6887
Epoch 16, Loss: 0.6886
Epoch 17, Loss: 0.6889
Epoch 18, Loss: 0.6888
Epoch 19, Loss: 0.6888
Epoch 20, Loss: 0.6883
RNN Results:
Accuracy: 0.5666
Precision: 0.5666
Recall: 1.0000
F1 Score: 0.7233


In [ ]:
# LSTM
lstm_model = LSTMModel(vocab_size)
train_model(lstm_model, train_loader)
print("LSTM Results:")
evaluate(lstm_model, X_test, y_test)

Epoch 1, Loss: 0.6875
Epoch 2, Loss: 0.6867
Epoch 3, Loss: 0.6511
Epoch 4, Loss: 0.3724
Epoch 5, Loss: 0.2334
Epoch 6, Loss: 0.1746
Epoch 7, Loss: 0.1380
Epoch 8, Loss: 0.1119
Epoch 9, Loss: 0.0918
Epoch 10, Loss: 0.0795
Epoch 11, Loss: 0.0663
Epoch 12, Loss: 0.0574
Epoch 13, Loss: 0.0500
Epoch 14, Loss: 0.0448
Epoch 15, Loss: 0.0423
Epoch 16, Loss: 0.0383
Epoch 17, Loss: 0.0388
Epoch 18, Loss: 0.0344
Epoch 19, Loss: 0.0341
Epoch 20, Loss: 0.0314
LSTM Results:
Accuracy: 0.9218
Precision: 0.9389
Recall: 0.9220
F1 Score: 0.9304


# Comparison of RNN and LSTM Models

In this assignment, both Recurrent Neural Network (RNN) and Long Short-Term Memory (LSTM) models were trained on the SST-2 dataset for sentiment classification. The performance of both models was evaluated using accuracy, precision, recall, and F1-score.

## 1. Performance Comparison

| Metric | RNN | LSTM |
|-------|------|------|
| Loss | ~0.688 (constant) | 0.0314 |
| Accuracy | 56.66% | 92.18% |
| Precision | 56.66% | 93.89% |
| Recall | 100% | 92.20% |
| F1 Score | 72.33% | 93.04% |

---

## 2. Analysis of RNN Performance

The RNN model showed poor learning behavior, as indicated by its nearly constant loss value (~0.688), which is close to the baseline loss for random guessing. The model predicted all samples as belonging to a single class, resulting in a recall of 1.0 but low precision and accuracy.

This behavior occurs due to the **vanishing gradient problem**, which prevents standard RNNs from learning long-term dependencies in text data.

---

## 3. Analysis of LSTM Performance

The LSTM model demonstrated strong learning capability, with loss decreasing steadily from ~0.68 to 0.03. It achieved high accuracy (92.18%) and F1-score (93.04%).

LSTM uses a gating mechanism (input, forget, output gates) that allows it to retain important information over long sequences, enabling better understanding of context and sentiment.

---

## 4. Key Improvements of LSTM over RNN

- **Better Learning:** LSTM converges while RNN fails to learn effectively  
- **Higher Accuracy:** Improved from 56.66% to 92.18%  
- **Balanced Predictions:** LSTM predicts both classes correctly  
- **Context Understanding:** Captures dependencies like "not good"  
- **Solves Vanishing Gradient Problem**

---

## 5. Conclusion

The LSTM model significantly outperforms the RNN model for sentiment analysis. Its ability to capture long-term dependencies and maintain contextual information makes it more suitable for NLP tasks.